# Meshek-ML: Perishable Inventory Optimization — Quickstart

This notebook walks through the full **two-stage pipeline** (forecast → optimize):
1. Set up the Colab environment and install dependencies
2. Mount Google Drive for data input/output
3. Configure pipeline parameters (data source, model, dates)
4. Generate or load demand data
5. Explore demand patterns, seasonality, and spoilage
6. Train a LightGBM demand forecast and evaluate metrics
7. Bridge: extract forecast-informed demand parameters for optimization
8. Train a PPO reinforcement learning agent to optimize ordering
9. Compare the RL agent against a classical newsvendor baseline
10. Visualize a single ordering episode

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yinonov/meshek-ml/blob/main/notebooks/colab_quickstart.ipynb)

**This notebook requires a fresh Colab runtime. Run cells 1–4 in order to set up the environment.**

## 1. Environment Setup

Install the package and its dependencies from the repository's `pyproject.toml`.
This pulls in simulation, forecasting, optimization, and experiment tracking extras.

In [ ]:
# Clone the repository (idempotent — skips if already cloned)
!git clone https://github.com/yinonov/meshek-ml.git 2>/dev/null || echo "Repository already cloned."
%cd meshek-ml

# Install the package with core + needed extras
# Note: we install simulation and tracking extras, plus forecasting/optimization
# deps directly (avoiding u8darts[all] which is heavy and unused in this notebook)
!pip install -q -e ".[simulation,tracking]"
!pip install -q lightgbm xgboost scikit-learn gymnasium "stable-baselines3[extra]" torch

# Verify installation
import meshek_ml
print(f"meshek-ml installed successfully")


In [ ]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"GPU available: {gpu_name} ({gpu_mem:.1f} GB)")
else:
    print("No GPU detected — CPU mode. PPO training will be slower but functional.")
    print("Tip: Runtime > Change runtime type > GPU to enable GPU acceleration.")

## 2. Google Drive Access

Mount Google Drive to read real sales data and save outputs persistently across sessions.
If running locally (not in Colab), this falls back to local directories.

In [ ]:
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')

    # Create a project output directory on Drive
    DRIVE_OUTPUT = "/content/drive/MyDrive/meshek-ml-outputs"
    os.makedirs(DRIVE_OUTPUT, exist_ok=True)
    print(f"Drive mounted. Output directory: {DRIVE_OUTPUT}")

    # Optional: path to real data (user sets this)
    DRIVE_DATA_INPUT = "/content/drive/MyDrive/meshek-ml-data"
    if os.path.exists(DRIVE_DATA_INPUT):
        print(f"Data input directory found: {DRIVE_DATA_INPUT}")
        print(f"Files: {os.listdir(DRIVE_DATA_INPUT)}")
    else:
        print(f"No data input directory at {DRIVE_DATA_INPUT} — using synthetic data only.")
except ImportError:
    print("Not running in Google Colab — skipping Drive mount.")
    DRIVE_OUTPUT = "./outputs"
    DRIVE_DATA_INPUT = "./data"
    os.makedirs(DRIVE_OUTPUT, exist_ok=True)
    print(f"Local output directory: {DRIVE_OUTPUT}")

## 3. Pipeline Parameters

One cell to control the entire workflow: data source, paths, dates, seed, and model.
Change these values and re-run all cells below.

In [ ]:
# === Pipeline Parameters ===
# Change these values to control the entire workflow.

# Data source: "synthetic" (generate data) or "csv"/"parquet" (load from path)
DATA_SOURCE = "synthetic"

# Path to real data file (only used when DATA_SOURCE is "csv" or "parquet")
DATA_PATH = None  # e.g., "/content/drive/MyDrive/meshek-ml-data/sales.csv"

# Simulation parameters (only used when DATA_SOURCE is "synthetic")
N_MERCHANTS = 3
SIM_START_DATE = "2024-01-01"
SIM_END_DATE = "2024-12-31"

# Forecasting parameters
MODEL_TYPE = "lightgbm"          # "lightgbm" or "xgboost"
TRAIN_END_DATE = "2024-09-30"    # Train on dates <= this, validate on dates after
FORECAST_HORIZON = 7             # Days ahead (used for context, not windowing in v1)

# Reproducibility
SEED = 42

## 4. Generate Synthetic Data


In [ ]:
from meshek_ml.simulation.generator import run_simulation
from meshek_ml.common.seed import set_global_seed

set_global_seed(SEED)

if DATA_SOURCE == "synthetic":
    df = run_simulation(
        n_merchants=N_MERCHANTS,
        start_date=SIM_START_DATE,
        end_date=SIM_END_DATE,
        seed=SEED,
    )
    print(f"Generated synthetic data: {df.shape[0]:,} rows")
elif DATA_SOURCE in ("csv", "parquet"):
    from meshek_ml.common.io import load_csv, load_parquet
    loader = load_csv if DATA_SOURCE == "csv" else load_parquet
    df = loader(DATA_PATH)
    print(f"Loaded {DATA_SOURCE} data from {DATA_PATH}: {df.shape[0]:,} rows")
else:
    raise ValueError(f"Unknown DATA_SOURCE: {DATA_SOURCE!r}. Use 'synthetic', 'csv', or 'parquet'.")

print(f"Merchants: {df['merchant_id'].nunique()}")
print(f"Products: {df['product'].nunique()}")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
df.head()

## 5. Explore the Data

In [ ]:
import matplotlib.pyplot as plt

# Pick one merchant to visualize
merchant_id = df["merchant_id"].unique()[0]
merchant_df = df[df["merchant_id"] == merchant_id].copy()

fig, ax = plt.subplots(figsize=(14, 5))
for product, group in merchant_df.groupby("product"):
    ax.plot(group["date"], group["realized_demand"], label=product, alpha=0.7)

ax.set_title(f"Daily Demand — Merchant: {merchant_id}")
ax.set_xlabel("Date")
ax.set_ylabel("Units demanded")
ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# Weekly demand pattern (averaged across all merchants/products)
df["day_of_week"] = df["date"].dt.day_name()
day_order = ["Sunday", "Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday"]
weekly = df.groupby("day_of_week")["realized_demand"].mean().reindex(day_order)

fig, ax = plt.subplots(figsize=(8, 4))
weekly.plot(kind="bar", ax=ax, color="steelblue")
ax.set_title("Average Demand by Day of Week")
ax.set_ylabel("Mean demand")
ax.set_xticklabels(day_order, rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
from meshek_ml.simulation.spoilage import weibull_quality

# Spoilage curves for different product types
days = np.arange(0, 15)
products_spoilage = {
    "Strawberries (shelf=3d)": {"shape": 2.0, "scale": 3.0},
    "Tomatoes (shelf=6d)": {"shape": 2.0, "scale": 6.0},
    "Oranges (shelf=14d)": {"shape": 2.0, "scale": 14.0},
    "Apples (shelf=30d)": {"shape": 2.0, "scale": 30.0},
}

fig, ax = plt.subplots(figsize=(10, 5))
for label, params in products_spoilage.items():
    quality = weibull_quality(days, **params)
    ax.plot(days, quality, label=label, linewidth=2)

ax.axhline(y=0.3, color="red", linestyle="--", alpha=0.5, label="Spoilage threshold")
ax.set_title("Product Quality Decay (Weibull Model)")
ax.set_xlabel("Age (days)")
ax.set_ylabel("Quality (0–1)")
ax.legend()
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

## 6. Demand Forecasting

Train a LightGBM model to predict daily demand per merchant-product.
The pipeline validates the data schema, engineers lag/rolling/calendar features,
splits by date (not random), trains, and evaluates.

In [ ]:
from meshek_ml.forecasting.pipeline import run_forecast_pipeline
from meshek_ml.forecasting.schema import normalize_simulation_data, validate_demand_schema, SchemaValidationError

# Prepare data for forecasting
forecast_df = df.copy()

# If synthetic, normalize column names to canonical schema
if DATA_SOURCE == "synthetic":
    forecast_df = normalize_simulation_data(forecast_df)
    print("Normalized simulation columns -> canonical schema")

# Validate schema (will fail fast on bad real data)
try:
    forecast_df = validate_demand_schema(forecast_df)
    print(f"Schema valid. Columns: {list(forecast_df.columns)}")
except SchemaValidationError as e:
    print(f"SCHEMA ERROR: {e}")
    print("Fix your data to have columns: date, merchant_id, product, quantity")
    raise

# Run the forecasting pipeline
print(f"\nTraining {MODEL_TYPE} model...")
print(f"Train period: <= {TRAIN_END_DATE}")
print(f"Validation period: > {TRAIN_END_DATE}")

metrics = run_forecast_pipeline(
    data=forecast_df,
    model_type=MODEL_TYPE,
    train_end_date=TRAIN_END_DATE,
    horizon=FORECAST_HORIZON,
    seed=SEED,
)

In [ ]:
import pandas as pd

metrics_df = pd.DataFrame([
    {"Metric": "MAE", "Value": metrics["mae"], "Description": "Mean Absolute Error"},
    {"Metric": "RMSE", "Value": metrics["rmse"], "Description": "Root Mean Squared Error"},
    {"Metric": "WMAPE", "Value": metrics["wmape"], "Description": "Weighted Mean Absolute Percentage Error"},
    {"Metric": "Pinball Loss", "Value": metrics["pinball_loss"], "Description": "Quantile loss (tau=0.5)"},
])

print(f"=== Forecast Evaluation ({MODEL_TYPE.upper()}) ===")
print(f"Train: <= {TRAIN_END_DATE} | Validate: > {TRAIN_END_DATE}")
print(f"Seed: {SEED}\n")
display(metrics_df.style.format({"Value": "{:.4f}"}).hide(axis="index"))

## 7. Forecast → Optimization Bridge

Use forecast predictions to estimate demand parameters for the inventory optimizer.
This is the **two-stage architecture**: forecast demand first, then optimize ordering.

The LightGBM forecast from Section 6 gives us predicted daily demand. We extract
the mean and standard deviation to configure the inventory environment and newsvendor baseline.

In [ ]:
# --- Two-Stage Bridge: Forecast → Optimization ---
# Use LightGBM predictions (not raw actuals) to estimate demand parameters

from meshek_ml.forecasting.pipeline import run_forecast_pipeline
from meshek_ml.forecasting.schema import normalize_simulation_data, validate_demand_schema
import numpy as np

# Prepare data
forecast_bridge_df = df.copy()
if DATA_SOURCE == "synthetic":
    forecast_bridge_df = normalize_simulation_data(forecast_bridge_df)
forecast_bridge_df = validate_demand_schema(forecast_bridge_df)

# Run pipeline with return_predictions=True to get LightGBM predictions
metrics, predictions = run_forecast_pipeline(
    data=forecast_bridge_df,
    model_type=MODEL_TYPE,
    train_end_date=TRAIN_END_DATE,
    seed=SEED,
    return_predictions=True,
)

# Derive demand parameters from LightGBM predictions (not raw actuals)
overall_mean = float(predictions.mean())
overall_std = float(predictions.std())

# Estimate NegBin dispersion: Var = mean + mean^2/dispersion
variance = overall_std ** 2
if variance > overall_mean:
    estimated_dispersion = overall_mean ** 2 / (variance - overall_mean)
else:
    estimated_dispersion = 10.0  # fallback for low-variance predictions

print("=== Forecast-Informed Optimization Parameters ===")
print(f"Source: LightGBM predictions on validation set ({len(predictions)} data points)")
print(f"Predicted demand mean: {overall_mean:.1f} units/day")
print(f"Predicted demand std:  {overall_std:.1f} units/day")
print(f"Estimated NegBin dispersion: {estimated_dispersion:.2f}")
print(f"\nThese feed into the PPO environment and newsvendor formula below.")

## 8. Train an RL Agent (PPO)


In [ ]:
from meshek_ml.optimization.env import PerishableInventoryEnv
from meshek_ml.optimization.rewards import CostParams

# Create environment — uses forecast-informed demand parameters from Section 7
costs = CostParams(
    selling_price=7.0,
    purchase_cost=5.0,
    holding_cost=0.1,
    waste_penalty=5.0,
    stockout_penalty=3.0,
)

env = PerishableInventoryEnv(
    max_shelf_life=7,
    max_order=50,
    demand_mean=overall_mean,
    demand_dispersion=estimated_dispersion,
    episode_length=90,
    costs=costs,
)

print(f"Observation space: {env.observation_space.shape}")
print(f"Action space: {env.action_space}")
print(f"Demand: mean={overall_mean:.1f}, dispersion={estimated_dispersion:.2f} (from forecast)")

In [ ]:
from meshek_ml.optimization.ppo_agent import train_ppo

# Train for 50k steps (~2 min on Colab GPU, ~5 min on CPU)
model = train_ppo(
    env,
    total_timesteps=50_000,
    learning_rate=3e-4,
    n_steps=2048,
)
print("Training complete!")

## 9. Evaluate: PPO vs Newsvendor Baseline


In [ ]:
from meshek_ml.optimization.newsvendor import optimal_order_negbin
from meshek_ml.optimization.evaluation import compute_inventory_metrics
import numpy as np


def run_episode(env, policy_fn, seed=0):
    """Run one episode and collect metrics."""
    obs, _ = env.reset(seed=seed)
    total_demand = 0
    total_sold = 0
    total_wasted = 0
    total_ordered = 0
    stockout_days = 0
    rewards = []
    history = []

    done = False
    while not done:
        action = policy_fn(obs)
        obs, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated

        total_demand += info["demand"]
        total_sold += info["sold"]
        total_wasted += info["wasted"]
        total_ordered += int(action[0])
        if info["unmet_demand"] > 0:
            stockout_days += 1
        rewards.append(reward)
        history.append(info | {"reward": reward, "order": int(action[0])})

    metrics = compute_inventory_metrics(
        total_demand=total_demand,
        total_sold=total_sold,
        total_wasted=total_wasted,
        total_ordered=total_ordered,
        total_stockout_events=stockout_days,
        n_days=env.episode_length,
    )
    metrics["total_reward"] = sum(rewards)
    return metrics, history


# PPO policy
def ppo_policy(obs):
    action, _ = model.predict(obs, deterministic=True)
    return action


# Newsvendor: fixed order quantity each day
nv_qty = optimal_order_negbin(
    mean_demand=overall_mean,
    dispersion=estimated_dispersion,
    underage_cost=costs.stockout_penalty,
    overage_cost=costs.waste_penalty,
)
print(f"Newsvendor optimal order: {nv_qty} units/day")


def newsvendor_policy(obs):
    return np.array([nv_qty], dtype=np.float32)


# Run evaluation episodes
n_eval = 10
ppo_results = []
nv_results = []
for i in range(n_eval):
    ppo_m, ppo_h = run_episode(env, ppo_policy, seed=SEED + i)
    nv_m, nv_h = run_episode(env, newsvendor_policy, seed=SEED + i)
    ppo_results.append(ppo_m)
    nv_results.append(nv_m)

print(f"Evaluated {n_eval} episodes for each policy")

In [ ]:
import pandas as pd

# Compare metrics
ppo_df = pd.DataFrame(ppo_results).mean()
nv_df = pd.DataFrame(nv_results).mean()

comparison = pd.DataFrame({"PPO Agent": ppo_df, "Newsvendor": nv_df})
comparison.index.name = "Metric"

# Highlight the key benchmarking metrics
print("=== Key Benchmarking Metrics ===")
for metric in ["fill_rate", "waste_rate", "stockout_frequency"]:
    ppo_val = comparison.loc[metric, "PPO Agent"]
    nv_val = comparison.loc[metric, "Newsvendor"]
    better = "PPO" if (ppo_val > nv_val if metric == "fill_rate" else ppo_val < nv_val) else "Newsvendor"
    print(f"  {metric}: PPO={ppo_val:.3f}, Newsvendor={nv_val:.3f} (better: {better})")

print(f"\n=== Full Comparison (avg over {n_eval} episodes, 90 days each) ===")
display(comparison.style.format("{:.3f}"))

## 10. Visualize a Single Episode

In [ ]:
# Run one detailed episode for each policy
_, ppo_history = run_episode(env, ppo_policy, seed=SEED)
_, nv_history = run_episode(env, newsvendor_policy, seed=SEED)

ppo_hist = pd.DataFrame(ppo_history)
nv_hist = pd.DataFrame(nv_history)

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# Inventory levels
axes[0].plot(ppo_hist["stock"], label="PPO", alpha=0.8)
axes[0].plot(nv_hist["stock"], label="Newsvendor", alpha=0.8)
axes[0].set_ylabel("Stock on hand")
axes[0].set_title("Inventory Levels Over 90 Days")
axes[0].legend()

# Waste
axes[1].bar(range(len(ppo_hist)), ppo_hist["wasted"], alpha=0.6, label="PPO", width=0.4)
axes[1].bar(
    [x + 0.4 for x in range(len(nv_hist))],
    nv_hist["wasted"],
    alpha=0.6,
    label="Newsvendor",
    width=0.4,
)
axes[1].set_ylabel("Units wasted")
axes[1].set_title("Daily Waste")
axes[1].legend()

# Cumulative reward
axes[2].plot(ppo_hist["reward"].cumsum(), label="PPO", alpha=0.8)
axes[2].plot(nv_hist["reward"].cumsum(), label="Newsvendor", alpha=0.8)
axes[2].set_ylabel("Cumulative reward")
axes[2].set_xlabel("Day")
axes[2].set_title("Cumulative Profit")
axes[2].legend()

plt.tight_layout()
plt.show()

## Next Steps

- **More training**: Increase `total_timesteps` to 200k+ for better convergence
- **Different products**: Try strawberries (shelf_life=3, harder!) or apples (shelf_life=30, easier)
- **Improve forecasting**: Try different `MODEL_TYPE` values, adjust `TRAIN_END_DATE`, or bring real data via `DATA_SOURCE = "csv"`
- **Federated learning**: Train models across multiple merchants without sharing raw data — see `meshek_ml.federated`